# 🚀 Hackathon Genius: od tematu do pitchu z Google ADK

W tym notebooku przejdziesz przez cały projekt `Hackathon Genius` przygotowany do pokazu dla studentów. Zobaczysz, jak działa orkiestracja agentów, jak agenty komunikują się przez `session.state`, jak korzystają z narzędzi oraz jak projekt łączy się z LM Proxy i ADK Web.

## Czego się nauczysz

- jak działa `SequentialAgent` jako orkiestrator,
- jaką rolę pełnią cztery wyspecjalizowane subagenty,
- jak wyglądają narzędzia i przepływ danych w projekcie,
- jak ładowana jest konfiguracja z `.env`,
- jak uruchomić demo w `adk web`.

## ‼️ Zanim zaczniesz

Ten notebook jest przygotowany pod **lokalne uruchomienie w VS Code / Jupyter**

Pracuj tak: 

1. uruchamiaj komórki po kolei,
2. miej aktywne środowisko `.venv`,
3. upewnij się, że plik `.env` istnieje w katalogu projektu,
4. jeśli chcesz wykonać pełne demo LLM, uruchom wcześniej LM Proxy na porcie `4000`.

Przykładowy temat do testów: `AI w edukacji`, `zdrowie psychiczne studentów`, `zrównoważony rozwój`.

## ⚙️ Sekcja 1: Setup

Najpierw znajdziemy katalog projektu i zainstalujemy zależności z `requirements.txt`.

In [10]:
import os
import subprocess
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'hackathon_genius').exists():
    for candidate in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
        if (candidate / 'hackathon_genius').exists():
            PROJECT_ROOT = candidate
            break

os.chdir(PROJECT_ROOT)
print(f'PROJECT_ROOT = {PROJECT_ROOT}')
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(PROJECT_ROOT / 'requirements.txt')])
print('✅ Zależności z requirements.txt są gotowe.')

PROJECT_ROOT = c:\swdtools\github\agh_conference
✅ Zależności z requirements.txt są gotowe.


### 1.2 Załaduj konfigurację z `.env`

Projekt używa lokalnego LM Proxy zgodnego z interfejsem OpenAI. W tej komórce sprawdzimy, czy plik `.env` jest widoczny i jakie najważniejsze wartości zawiera.

In [11]:
import os
from dotenv import load_dotenv

env_path = PROJECT_ROOT / '.env'
load_dotenv(env_path, override=False)

def _mask(value: str | None) -> str | None:
    if value is None:
        return None
    if len(value) <= 4:
        return '*' * len(value)
    return value[:2] + '***' + value[-2:]

config_preview = {
    'env_path': str(env_path),
    'env_exists': env_path.exists(),
    'OPENAI_API_BASE': os.getenv('OPENAI_API_BASE'),
    'LM_PROXY_BASE_URL': os.getenv('LM_PROXY_BASE_URL'),
    'LM_PROXY_MODEL': os.getenv('LM_PROXY_MODEL'),
    'OPENAI_API_KEY': _mask(os.getenv('OPENAI_API_KEY')),
    'LM_PROXY_API_KEY': _mask(os.getenv('LM_PROXY_API_KEY')),
}
config_preview

{'env_path': 'c:\\swdtools\\github\\agh_conference\\.env',
 'env_exists': True,
 'OPENAI_API_BASE': 'http://localhost:4000/openai/v1',
 'LM_PROXY_BASE_URL': None,
 'LM_PROXY_MODEL': 'openai/gpt-4.1',
 'OPENAI_API_KEY': 'lo***xy',
 'LM_PROXY_API_KEY': None}

## 🗂️ Sekcja 2: Struktura projektu

Zobaczmy najpierw, jakie pliki składają się na demo.

In [12]:
from pathlib import Path

IGNORE = {'.venv', '__pycache__'}

def show_tree(root: Path, max_depth: int = 2, prefix: str = ''):
    root = Path(root)
    if max_depth < 0:
        return
    entries = sorted(root.iterdir(), key=lambda item: (item.is_file(), item.name.lower()))
    for entry in entries:
        if entry.name in IGNORE:
            continue
        print(f'{prefix}- {entry.name}')
        if entry.is_dir() and max_depth > 0:
            show_tree(entry, max_depth=max_depth - 1, prefix=prefix + '  ')

show_tree(PROJECT_ROOT, max_depth=2)

- hackathon_genius
  - sub_agents
    - __init__.py
    - idea_agent.py
    - pitch_agent.py
    - tech_stack_agent.py
    - timeline_agent.py
  - tools
    - __init__.py
    - hackathon_tools.py
  - __init__.py
  - agent.py
  - model_config.py
- .env
- ARCHITEKTURA_DEMO_PL.md
- hackathon-genius-od-tematu-do-pitchu.ipynb
- requirements.txt


## 🧭 Sekcja 3: Architektura wieloagentowa

Poniżej masz uproszczony rysunek ASCII. To ten sam przepływ, który potem zobaczysz w ADK Web jako osobne kroki i logi.

```text
+---------------------+
| Student / użytkownik |
+----------+----------+
           |
           v
+---------------------+
|    ADK Web Client   |
+----------+----------+
           |
           v
+-------------------------------------------+
| HackathonGenius                            |
| SequentialAgent / orkiestrator             |
+----------+-------------+-------------+-----+
           |             |             |
           v             v             v
     +-----------+ +--------------+ +-------------+ +-----------+
     | IdeaAgent | | TechStack    | | Timeline    | | PitchAgent|
     +-----+-----+ +------+-------+ +------+------ +-----+-----+
           |              |                |              |
           v              v                v              v
  browse_hackathon_   get_tech_      get_sprint_    get_pitch_
       themes        stack_library     templates     frameworks

             +-----------------------------------------+
             |              session.state              |
             +-----------------------------------------+
                               |
                               v
                    +------------------------+
                    |    model_config.py     |
                    +-----------+------------+
                                |
                                v
             +-------------------------------------------+
             | LM Proxy for VS Code                      |
             | localhost:4000/openai/v1                  |
             +-------------------+-----------------------+
                                 |
                                 v
                    VS Code Language Model API
                                 |
                                 v
                         Wybrany model LLM
```

In [13]:
from hackathon_genius.agent import root_agent

root_summary = {
    'name': root_agent.name,
    'type': type(root_agent).__name__,
    'description': root_agent.description,
    'sub_agents': [agent.name for agent in root_agent.sub_agents],
}
root_summary

{'name': 'HackathonGenius',
 'type': 'SequentialAgent',
 'description': 'Wieloagentowy pipeline, który zamienia temat hackathonu w kompletny plan działania: 3 pomysły na projekt -> stack technologiczny -> plan sprintu -> pitch.',
 'sub_agents': ['IdeaAgent', 'TechStackAgent', 'TimelineAgent', 'PitchAgent']}

In [14]:
import pandas as pd
from IPython.display import display

from hackathon_genius.sub_agents.idea_agent import idea_agent
from hackathon_genius.sub_agents.tech_stack_agent import tech_stack_agent
from hackathon_genius.sub_agents.timeline_agent import timeline_agent
from hackathon_genius.sub_agents.pitch_agent import pitch_agent

agents = [idea_agent, tech_stack_agent, timeline_agent, pitch_agent]
rows = [
    {
        'agent': agent.name,
        'output_key': getattr(agent, 'output_key', None),
        'description': agent.description,
    }
    for agent in agents
]
display(pd.DataFrame(rows))

,agent,output_key,description
0,IdeaAgent,hackathon_ideas,Generuje 3 kreatywne i innowacyjne pomysły na ...
1,TechStackAgent,tech_stacks,Rekomenduje najlepszy stack technologiczny dla...
2,TimelineAgent,sprint_plan,Tworzy szczegółowy i realistyczny plan sprintu...
3,PitchAgent,elevator_pitch,Pisze przekonujący i zapadający w pamięć 1-min...


In [ ]:
import pandas as pd
from IPython.display import display

pd.set_option('display.max_colwidth', None)

# Ustaw liczbę znaków do skrótu (np. 180) albo None, aby pokazać pełną instrukcję.
MAX_CHARS = 180

def normalize_text(value: object) -> str:
    return ' '.join(str(value).split())

def shorten_to_chars(text: str, max_chars: int | None) -> str:
    if max_chars is None:
        return text
    if max_chars <= 0:
        return ''
    if len(text) <= max_chars:
        return text
    if max_chars <= 3:
        return text[:max_chars]
    return text[:max_chars - 3].rstrip() + '...'

prompt_preview = [
    {
        'agent': agent.name,
        'instrukcja': shorten_to_chars(normalize_text(agent.instruction), MAX_CHARS),
    }
    for agent in agents
]

display(pd.DataFrame(prompt_preview))

,agent,pelna_instrukcja
0,IdeaAgent,"Jesteś kreatywnym agentem do generowania pomysłów hackathonowych. Masz talent do dostrzegania ciekawych szans produktowych i technologicznych. Zawsze odpowiadaj wyłącznie po polsku. Nawet jeśli narzędzie zwróci dane po angielsku, końcową odpowiedź sformatuj po polsku. Twoje zadanie: 1. Weź temat hackathonu podany przez użytkownika. 2. Wywołaj narzędzie `browse_hackathon_themes` z tym tematem, aby pobrać inspiracje z poprzednich zwycięskich projektów. 3. Na podstawie wyniku narzędzia ORAZ własnej kreatywności wygeneruj dokładnie 3 unikalne pomysły na projekt. Dla KAŻDEGO pomysłu podaj: - 💡 **Nazwa projektu**: chwytliwy i zapamiętywalny tytuł - 🎯 **Rozwiązywany problem**: jedno zdanie opisujące realny problem użytkownika - 🚀 **Kluczowa funkcja**: najważniejsza rzecz, którą robi produkt - 🌟 **Efekt wow**: co wyróżnia ten pomysł na tle istniejących rozwiązań Każdy pomysł ma być zwięzły, ale ekscytujący. Myśl o tym, co zrobi wrażenie na jury hackathonu. Użyj numerowanej listy: Pomysł 1, Pomysł 2, Pomysł 3. Po przedstawieniu 3 pomysłów dodaj krótką linię: ""Przekazuję pomysły do agenta TechStackAgent..."""
1,TechStackAgent,"Jesteś doświadczonym architektem oprogramowania, który specjalizuje się w szybkim budowaniu projektów na hackathony. Zawsze odpowiadaj wyłącznie po polsku. Nawet jeśli narzędzie zwróci dane po angielsku, końcową odpowiedź sformatuj po polsku. IdeaAgent wygenerował już 3 pomysły na projekt. Masz je w swoim kontekście. Twoje zadanie: 1. Dla każdego z 3 pomysłów określ typ projektu (np. web app, mobile app, data, IoT). 2. Dla każdego pomysłu wywołaj narzędzie `get_tech_stack_library` z odpowiednim typem projektu. 3. Przy wywołaniu narzędzia używaj angielskich kategorii rozpoznawanych przez bibliotekę: `web`, `mobile`, `data` albo `iot`. 4. Na podstawie wyniku narzędzia zarekomenduj najlepszy stack do zbudowania tego projektu na hackathonie. Dla KAŻDEGO pomysłu podaj: - 🏗️ **Pomysł**: powtórz nazwę projektu - ⚡ **Dlaczego ten stack**: jedno zdanie, dlaczego to dobre rozwiązanie na hackathon - 🖥️ **Frontend**: wybierz 1 rekomendację i krótko uzasadnij - 🔧 **Backend**: wybierz 1 rekomendację i krótko uzasadnij - 🤖 **AI/ML**: wybierz 1 rekomendację i krótko uzasadnij - 🗃️ **Baza danych**: wybierz 1 rekomendację i krótko uzasadnij - ☁️ **Hosting**: wybierz 1 rekomendację Priorytet mają technologie, które są szybkie we wdrożeniu, mają dobry darmowy plan i dobrze nadają się do dema. Na końcu dodaj linię: ""Stacki technologiczne gotowe. Przekazuję wynik do agenta TimelineAgent..."""
2,TimelineAgent,"Jesteś doświadczonym mentorem hackathonowym, który wspierał wiele zwycięskich zespołów. Zawsze odpowiadaj wyłącznie po polsku. Nawet jeśli dane wejściowe lub wynik narzędzia są po angielsku, odpowiedź końcową przygotuj po polsku. Poprzednie agenty wygenerowały już 3 pomysły wraz ze stackami technologicznymi. Masz je w swoim kontekście. Twoje zadanie: 1. Wybierz NAJBARDZIEJ WYKONALNY pomysł, czyli taki, który studenci realnie dowiozą na hackathonie. Krótko wyjaśnij, dlaczego go wybrałeś. 2. Wywołaj narzędzie `get_sprint_templates` z parametrami team_size=4 i duration_days=2, czyli dla standardowego 2-dniowego hackathonu studenckiego. 3. Dostosuj zwrócony szablon do konkretnego planu dla wybranego projektu. Plan sprintu powinien zawierać: - 📋 **Wybrany projekt**: nazwa i jednozdaniowe uzasadnienie, dlaczego jest najbardziej wykonalny - 👥 **Role w zespole**: przypisz 4 role (np. Frontend Developer, Backend Developer, AI Engineer, Designer/PM) - 🗓️ **Plan etapami**: dla każdego etapu z szablonu podaj: - blok czasowy - kto za co odpowiada - konkretny rezultat lub milestone - ⚠️ **Ryzyka**: 2-3 rzeczy, które mogą pójść źle, oraz jak im zapobiec - 🏁 **Definicja sukcesu**: jak powinno wyglądać MVP gotowe do dema Na końcu dodaj linię: ""Plan sprintu gotowy. Przekazuję go do agenta PitchAgent, który przygotuje prezentację..."""
3,PitchAgent,"Jesteś profesjonalnym coachem prezentacji i storytellingu, kt

## 🛠️ Sekcja 4: Narzędzia agentów

Każdy agent ma własne narzędzie. To ważne, bo dzięki temu w ADK Web można pokazać studentom, że agent nie jest tylko jednym dużym promptem, ale współpracuje z funkcjami i zewnętrznym kontekstem.

In [16]:
from IPython.display import JSON, display

from hackathon_genius.tools.hackathon_tools import (
    browse_hackathon_themes,
    get_pitch_frameworks,
    get_sprint_templates,
    get_tech_stack_library,
)

tool_demo = {
    'browse_hackathon_themes': browse_hackathon_themes('AI w edukacji'),
    'get_tech_stack_library': get_tech_stack_library('web'),
    'get_sprint_templates': get_sprint_templates(team_size=4, duration_days=2),
    'get_pitch_frameworks': get_pitch_frameworks('judges'),
}
display(JSON(tool_demo, expanded=False))

<IPython.core.display.JSON object>

## 🤖 Sekcja 5: Model i LM Proxy

W tym projekcie wszystkie agenty korzystają ze wspólnej konfiguracji z pliku `model_config.py`. To tutaj normalizowany jest model i adres `api_base`, tak aby kod był zgodny z lokalnym LM Proxy w VS Code.

In [17]:
from hackathon_genius.model_config import (
    _normalize_api_base,
    _normalize_model_name,
    build_demo_model,
)

api_base_raw = os.getenv('LM_PROXY_BASE_URL', os.getenv('OPENAI_API_BASE', 'http://localhost:4000/openai/v1'))
model_raw = os.getenv('LM_PROXY_MODEL', 'gpt-4.1')
demo_model = build_demo_model()

model_debug = {
    'api_base_raw': api_base_raw,
    'api_base_normalized': _normalize_api_base(api_base_raw),
    'model_raw': model_raw,
    'model_normalized': _normalize_model_name(model_raw, _normalize_api_base(api_base_raw)),
    'demo_model_class': type(demo_model).__name__,
    'demo_model_name': getattr(demo_model, 'model', None),
}
model_debug

{'api_base_raw': 'http://localhost:4000/openai/v1',
 'api_base_normalized': 'http://localhost:4000/openai/v1',
 'model_raw': 'openai/gpt-4.1',
 'model_normalized': 'openai/gpt-4.1',
 'demo_model_class': 'LiteLlm',
 'demo_model_name': 'openai/gpt-4.1'}

## ▶️ Sekcja 6: Jak uruchomić demo w ADK Web

Najwygodniejsze demo dla studentów zobaczysz w przeglądarce przez `adk web`. Wtedy widać kolejność agentów, wywołania narzędzi i logi przebiegu.

In [18]:
run_plan = {
    'katalog': str(PROJECT_ROOT),
    'komenda': 'adk web',
    'adres_domyslny': 'http://localhost:8000',
    'agent_do_wyboru': 'hackathon_genius',
    'przykladowe_prompty': [
        'AI w edukacji',
        'zdrowie psychiczne studentów',
        'zrównoważony rozwój na kampusie',
    ],
}
run_plan

{'katalog': 'c:\\swdtools\\github\\agh_conference',
 'komenda': 'adk web',
 'adres_domyslny': 'http://localhost:8000',
 'agent_do_wyboru': 'hackathon_genius',
 'przykladowe_prompty': ['AI w edukacji',
  'zdrowie psychiczne studentów',
  'zrównoważony rozwój na kampusie']}

### Co pokażesz studentom podczas live demo

1. Student wpisuje jeden temat hackathonu.
2. `IdeaAgent` generuje trzy pomysły.
3. `TechStackAgent` dobiera technologie do każdego pomysłu.
4. `TimelineAgent` wybiera najbardziej wykonalny projekt i tworzy plan sprintu.
5. `PitchAgent` przygotowuje końcowy pitch dla jury.
6. W logach ADK Web widać cały przepływ i narzędzia.

## 🧪 Sekcja 7: Ćwiczenia dla studentów

Spróbuj wykonać przynajmniej dwa z poniższych zadań:

1. Zmień instrukcję `IdeaAgent`, aby generował nie 3, ale 5 pomysłów.
2. Dodaj nowy motyw do `browse_hackathon_themes`, np. `smart city` albo `cybersecurity`.
3. Zmień `PitchAgent`, aby tworzył pitch dla `students` zamiast `judges`.
4. Zmień `team_size` lub `duration_days` w `TimelineAgent` i porównaj plan sprintu.
5. Dodaj piątego subagenta, który tworzy listę potencjalnych ryzyk biznesowych lub technicznych.

## ✅ Podsumowanie

Ten projekt dobrze nadaje się do pokazów dydaktycznych, bo rozbija jedno duże zadanie na kilka prostszych kroków i pozwala studentom zobaczyć różnicę między:

- orkiestratorem,
- wyspecjalizowanymi agentami,
- narzędziami,
- stanem sesji,
- konfiguracją modelu i runtime'em ADK.

Jeśli chcesz rozwinąć ten notebook, możesz dodać osobną sekcję z testami, obserwowalnością lub porównaniem kilku modeli przez LM Proxy.